# 04. One-stage anchor-free detector: YOLO11
YOLO11m (Ultralytics, COCO-pretrained). Anchor-free head: every feature-map location predicts class scores and
distances to the four box sides (DFL + CIoU loss). Positives are chosen by the Task-Aligned Assigner (TOOD):
for each object, the top-10 locations inside the box ranked by score^0.5 * IoU^6. Inference uses NMS.
Budget: 1.25 GPU-hours per run (as one torchvision run); Ultralytics fits the number of epochs and the lr schedule
into it. Same frames and validation as before; Ultralytics default augmentation (mosaic, HSV, scale, flip).

In [1]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
%load_ext autoreload
%autoreload 2
import sys, json, subprocess
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())))
%pip install -q -U ultralytics

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import PROC, ROOT, PROJECT, CLASSES, seed_everything, write_submission, load_results
import yolo_utils as yu

seed_everything()
VAL_GT = PROC / "val_gt.json"
test_df = pd.read_csv(ROOT / "test_images.csv")
COMP = "multiple-object-video-detection-hw-2-ucu-ai-2026"
BUDGET_H = 1.25

Note: you may need to restart the kernel to use updated packages.


## 1. How many locations can be assigned to a small object?
Without anchors, an object can only be learned from feature-map locations whose centre lies inside its box.
We count them for every training box (at input 1280) for stride 8 (P3, the finest YOLO11 level) and stride 4 (a P2 head).

In [2]:
# number of grid centres ((i + 0.5) * stride) inside each box, boxes rescaled to long side 1280
rows = []
for f in (PROC / "s10" / "labels" / "train").glob("*.txt"):
    lab = np.loadtxt(f, ndmin=2).reshape(-1, 5)
    W, H = 1280, 720                                 # all frames are ~16:9
    x1, x2 = (lab[:, 1] - lab[:, 3] / 2) * W, (lab[:, 1] + lab[:, 3] / 2) * W
    y1, y2 = (lab[:, 2] - lab[:, 4] / 2) * H, (lab[:, 2] + lab[:, 4] / 2) * H
    row = {"sqrt_area": np.sqrt((x2 - x1) * (y2 - y1))}
    for s in (4, 8):
        nx = np.floor(x2 / s - 0.5) - np.ceil(x1 / s - 0.5) + 1
        ny = np.floor(y2 / s - 0.5) - np.ceil(y1 / s - 0.5) + 1
        row[f"points_s{s}"] = np.clip(nx, 0, None) * np.clip(ny, 0, None)
    rows.append(pd.DataFrame(row))
pts = pd.concat(rows, ignore_index=True)
pts["size at 1280"] = pd.cut(pts.sqrt_area, [0, 8, 16, 32, 64, 1e9], labels=["<8", "8-16", "16-32", "32-64", ">64"])
display(pts.groupby("size at 1280", observed=True).agg(
    objects=("sqrt_area", "size"),
    median_points_stride8=("points_s8", "median"), share_under_10_points_s8=("points_s8", lambda v: (v < 10).mean()),
    median_points_stride4=("points_s4", "median"), share_under_10_points_s4=("points_s4", lambda v: (v < 10).mean())).round(2))

,objects,median_points_stride8,share_under_10_points_s8,median_points_stride4,share_under_10_points_s4
size at 1280,,,,,
<8,636,1.0,1.00,3.0,1.00
8-16,10176,3.0,1.00,12.0,0.33
16-32,49093,9.0,0.57,36.0,0.00
32-64,38731,27.0,0.00,108.0,0.00
>64,12218,105.0,0.00,416.0,0.00


## 2. Baseline: YOLO11m at 1280

In [2]:
# If the kernel dies, just re-run: the run resumes from runs/yolo_base/weights/last.pt
YOLO_BASE_CFG = dict(model="yolo11m", head="P3-P5", box_loss="ciou+dfl", assigner="TAL", budget_h=BUDGET_H, batch=4)
w_base = yu.train_ultra("yolo_base", "yolo11m.pt", imgsz=1280, batch=4, hours=BUDGET_H)
res_base, _ = yu.evaluate_ultra("yolo_base", w_base, VAL_GT, YOLO_BASE_CFG, imgsz=1280)
display(pd.Series(res_base).to_frame().T)

yolo_base: already trained, using /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/runs/yolo_base/weights/best.pt


,name,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AP_person,AP_bicycle,AP_car,...,ms_per_image,epochs_done,train_hours,imgsz,model,head,box_loss,assigner,budget_h,batch
0,yolo_base,0.2414,0.4676,0.2168,0.1333,0.2221,0.3894,0.2988,0.2223,0.5118,...,26.8,26,1.25,1280,yolo11m,P3-P5,ciou+dfl,TAL,1.25,4


In [3]:
# Test predictions -> Kaggle (checks the val -> public relation and the official baseline 0.2216)
from ultralytics import YOLO
test_preds, _ = yu.predict_ultra(YOLO(str(w_base)), yu.test_items(test_df), imgsz=1280)
sub_path = PROJECT / "submissions" / "yolo_base.csv"
write_submission(test_preds, test_df, sub_path)
subprocess.run(["kaggle", "competitions", "submit", "-c", COMP, "-f", str(sub_path),
                "-m", f"YOLO11m 1280, val mAP {res_base['mAP']:.4f}"], check=True)

100%|██████████| 45.3M/45.3M [00:16<00:00, 2.87MB/s]


Submission ref: 58084157
4 submissions remaining today.
Successfully submitted to Multiple Object Video Detection | HW2 | UCU AI 2026

CompletedProcess(args=['kaggle', 'competitions', 'submit', '-c', 'multiple-object-video-detection-hw-2-ucu-ai-2026', '-f', '/home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/submissions/yolo_base.csv', '-m', 'YOLO11m 1280, val mAP 0.2414'], returncode=0)

## 3. Experiment 1: input resolution 1536
01_data, section 1.2: at long side 1280, 9.7 % of objects are smaller than 16 px; at 1536 only 4.8 %.
Same 1.25 h budget, so the larger input gets fewer epochs: this tests whether resolution is worth its cost.

In [4]:
YOLO_1536_CFG = {**YOLO_BASE_CFG, "batch": 4}
w_1536 = yu.train_ultra("yolo_1536", "yolo11m.pt", imgsz=1536, batch=4, hours=BUDGET_H)
res_1536, _ = yu.evaluate_ultra("yolo_1536", w_1536, VAL_GT, YOLO_1536_CFG, imgsz=1536)

Ultralytics 8.4.204 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1536, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic

[W1107 09:31:13.328131118 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 25559040, total: 8082096128).
[W1107 09:31:13.328883825 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 27656192, total: 8082096128).


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 934.3±372.3 MB/s, size: 483.7 KB)
train: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/train.cache... 2446 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2446/2446 1.1Git/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 945.6±466.4 MB/s, size: 459.6 KB)
val: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/val.cache... 420 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 420/420 251.7Mit/s 0.0s
: 0% ──────────── 0/612  0.9s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      6.44G      1.646      1.155      1.196         89       1536: 100% ━━━━━━━━━━━━ 1223/1223 4.8it/s 4:170.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 105/105 9.7it/s 10.8s<0.2s
                   all        420      20624      0.525     

## 4. Experiment 2: extra high-resolution P2 head
`configs/yolo11m-p2.yaml`: YOLO11m plus a stride-4 detection level (structure as in Ultralytics yolov8-p2.yaml).
Section 1 shows how few stride-8 locations fall inside the smallest objects; stride 4 gives 4x more.
Backbone and the first part of the neck load COCO weights from yolo11m.pt; the new levels start from scratch.
Trained at 1280 to separate the effect of the head from the effect of resolution.

In [ ]:
YOLO_P2_CFG = {**YOLO_BASE_CFG, "head": "P2-P5"}
w_p2 = yu.train_ultra("yolo_p2", PROJECT / "configs" / "yolo11m-p2.yaml", pretrained="yolo11m.pt",
                      imgsz=1280, batch=2, hours=BUDGET_H)
res_p2, _ = yu.evaluate_ultra("yolo_p2", w_p2, VAL_GT, YOLO_P2_CFG, imgsz=1280)

Transferred 427/803 items from pretrained weights
Ultralytics 8.4.204 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

[W1107 10:59:21.000781560 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 39845888, total: 8082096128).
[W1107 10:59:21.018041899 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 39845888, total: 8082096128).
[W1107 10:59:21.024280206 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 39845888, total: 8082096128).


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8359.0±4110.8 MB/s, size: 483.7 KB)
train: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/train.cache... 2446 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2446/2446 932.7Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4531.4±327.6 MB/s, size: 459.6 KB)
val: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/val.cache... 420 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 420/420 176.2Mit/s 0.0s
: 0% ──────────── 0/612  1.0s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      6.42G      6.241      13.25      4.257         49       1280: 0% ──────────── 0/1223  0.9sWARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=2 and max_num_obj=180; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
      1/100      6.33G      5

[W1107 10:59:22.455112916 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 31588352, total: 8082096128).
[W1107 10:59:22.456402136 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 31588352, total: 8082096128).
[W1107 10:59:22.476431545 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 31588352, total: 8082096128).


      1/100      6.47G      4.925      7.828      4.298        258       1280: 0% ──────────── 2/1223 1.9it/s 1.4s<10:48

[W1107 10:59:22.730536415 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 31260672, total: 8082096128).
[W1107 10:59:22.734819251 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 31260672, total: 8082096128).


... (34 outputs left out)

      1/100      6.47G      3.934      5.067      4.399         39       1280: 4% ╸─────────── 60/1223 3.8it/s 17.2s<5:04WARNING ⚠️ CUDA out of memory with batch=2. Reducing to batch=1 and retrying (2/3).


[W1107 10:59:38.774120945 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 36503552, total: 8082096128).
[W1107 10:59:38.797912247 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 34406400, total: 8082096128).
[W1107 10:59:38.800072456 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 34406400, total: 8082096128).
[W1107 10:59:38.812260430 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 34406400, total: 8082096128).
[W1107 10:59:38.813768434 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 36503552, total: 8082096128).


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6282.8±2163.5 MB/s, size: 436.7 KB)
train: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/train.cache... 2446 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2446/2446 932.7Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4285.1±589.8 MB/s, size: 452.9 KB)
val: Scanning /home/julia/Documents/Yuliia/UCU/3/Artificial_intelligence/hw2/hw2-ai/data/proc/s10/labels/val.cache... 420 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 420/420 195.7Mit/s 0.0s
      1/100      6.47G      3.934      5.067      4.399         39       1280: 4% ╸─────────── 61/1223 3.3it/s 17.7s<5:55

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      6.22G      3.726      4.363      4.391         87       1280: 2% ──────────── 65/2446 7.3it/s 9.8s<5:27


KeyboardInterrupt: 

In [ ]:
cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_image", "epochs_done"]
names = ["frcnn_giou", "retina_focal", "yolo_base", "yolo_1536", "yolo_p2"]
tab = load_results().set_index("name").reindex(names)[cols]
display(tab)
tab[["AP_small", "AP_medium", "AP_large"]].plot.bar(rot=0, figsize=(9, 4), title="AP by object size: anchor-based vs anchor-free")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for n in ["yolo_base", "yolo_1536", "yolo_p2"]:
    h = yu.history(n); ax.plot(h["time"] / 3600, h["metrics/mAP50-95(B)"], marker=".", label=n)
ax.set_xlabel("training time, h"); ax.set_ylabel("Ultralytics val mAP50-95")
ax.set_title("YOLO runs under the same time budget"); ax.legend(); plt.show()